# 第8章: ニューラルネット

第7章で取り組んだポジネガ分類を題材として、ニューラルネットワークで分類モデルを実装する。なお、この章ではPyTorchやTensorFlow、JAXなどの深層学習フレームワークを活用せよ。

# Chương 8: Mạng nơ-ron (Neural Network)

Trong chương này, bạn sẽ xây dựng mô hình phân loại cảm xúc (positive/negative) giống chương 7, nhưng thay Logistic Regression bằng Neural Network.

Bạn được yêu cầu sử dụng các framework deep learning như:

- PyTorch
- TensorFlow
- JAX

## 70. 単語埋め込みの読み込み

事前学習済み単語埋め込みを活用し、$|V| \times d_\mathrm{emb}$ の単語埋め込み行列$\pmb{E}$を作成せよ。ここで、$|V|$は単語埋め込みの語彙数、$d_\mathrm{emb}$は単語埋め込みの次元数である。ただし、単語埋め込み行列の先頭の行ベクトル$\pmb{E}_{0,:}$は、将来的にパディング（`<PAD>`）トークンの埋め込みベクトルとして用いたいので、ゼロベクトルとして予約せよ。ゆえに、$\pmb{E}$の2行目以降に事前学習済み単語埋め込みを読み込むことになる。

もし、Google Newsデータセットの[学習済み単語ベクトル](https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing)（300万単語・フレーズ、300次元）を全て読み込んだ場合、$|V|=3000001, d_\mathrm{emb}=300$になるはずである（ただ、300万単語の中には、殆ど用いられない稀な単語も含まれるので、語彙を削減した方がメモリの節約になる）。

また、単語埋め込み行列の構築と同時に、単語埋め込み行列の各行のインデックス番号（トークンID）と、単語（トークン）への双方向の対応付けを保持せよ。


## 70. Đọc word embedding

Hãy tải pre-trained word embeddings và xây dựng:

- ma trận embedding W

W∈R
V×D

Trong đó:

- V: số lượng từ trong vocabulary
- D: số chiều embedding

Quy tắc quan trọng:
- Hàng đầu tiên của ma trận W dành cho token đặc biệt:
```
<PAD>
```
- vector của nó phải là vector 0:
  - W0=0

- từ hàng thứ 2 trở đi mới chứa embedding pretrained

Lưu ý thực tế

Nếu dùng Google News embeddings:

- ~3 triệu từ
- 300 chiều

⇒ rất nặng bộ nhớ → nên giảm vocabulary

Ngoài ra phải lưu:

- token → id
- id → token

(2 chiều mapping)

In [44]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [45]:
!pip install gensim

In [46]:
!wget https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
!unzip SST-2.zip

--2026-06-18 06:14:28--  https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.163.189.51, 3.163.189.108, 3.163.189.96, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.163.189.51|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7439277 (7.1M) [application/zip]
Saving to: ‘SST-2.zip.1’

SST-2.zip.1         100%[===================>]   7.09M  --.-KB/s    in 0.08s   

2026-06-18 06:14:28 (88.7 MB/s) - ‘SST-2.zip.1’ saved [7439277/7439277]

Archive:  SST-2.zip
replace SST-2/dev.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [47]:
import pandas as pd

train_df = pd.read_csv("SST-2/train.tsv",sep="\t")

dev_df = pd.read_csv("SST-2/dev.tsv",sep="\t")

print(len(train_df))
print(len(dev_df))

67349
872


In [48]:
vocab = set()

for text in train_df["sentence"]:
    vocab.update(text.split())

for text in dev_df["sentence"]:
    vocab.update(text.split())

print("Vocabulary size:", len(vocab))

Vocabulary size: 15756


In [49]:
from gensim.models import KeyedVectors

google_model = KeyedVectors.load_word2vec_format("/content/drive/MyDrive/Colab Notebooks/nlp100/data/GoogleNews-vectors-negative300.bin.gz",
    binary=True
)

In [50]:
google_model["cat"][:10]

array([ 0.0123291 ,  0.20410156, -0.28515625,  0.21679688,  0.11816406,
        0.08300781,  0.04980469, -0.00952148,  0.22070312, -0.12597656],
      dtype=float32)

In [51]:
# reduce vocab

available_words = []

for word in vocab:
    if word in google_model:
        available_words.append(word)

print(len(available_words))

13068


In [52]:
# word2id - id2word

word2id = {"<PAD>": 0}
id2word = {0: "<PAD>"}

for idx, word in enumerate(available_words,start=1):
    word2id[word] = idx
    id2word[idx] = word

print(word2id["movie"])
print(id2word[word2id["movie"]])

10846
movie


In [53]:
# embedding matrix
import numpy as np

dim = google_model.vector_size

embedding_matrix = np.zeros((len(word2id), dim), dtype=np.float32)

In [54]:
for word, idx in word2id.items():
    if word == "<PAD>":
        continue

    embedding_matrix[idx] = google_model[word]

embedding_matrix.shape

(13069, 300)

In [55]:
import torch

embedding_matrix = torch.tensor(embedding_matrix)

torch.save(embedding_matrix,"embedding_matrix.pt")
torch.save(word2id,"word2id.pt")
torch.save(id2word,"id2word.pt")

## 71. データセットの読み込み

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip) をダウンロードし、訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、全てのテキストをトークンID列に変換せよ。このとき、単語埋め込みの語彙でカバーされていない単語は無視し、トークン列に含めないことにせよ。また、テキストの全トークンが単語埋め込みの語彙に含まれておらず、空のトークン列となってしまう事例は、訓練セットおよび開発セットから削除せよ（このため、第7章の実験で得られた正解率と比較できなくなることに注意せよ）。

事例の表現方法は任意でよいが、例えば"contains no wit , only labored gags"がネガティブに分類される事例は、次のような辞書オブジェクトで表現すればよい。

```
{'text': 'contains no wit , only labored gags',
 'label': tensor([0.]),
 'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])}
```

この例では、`text`はテキスト、`label`は分類ラベル（ポジティブなら`tensor([1.])`、ネガティブなら`tensor([0.])`）、`input_ids`はテキストのトークン列をID列で表現している。

## 71. Đọc dataset

Tải bộ dữ liệu:

- GLUE
- SST-2

Xử lý:
- train.tsv
- dev.tsv

Chuyển mỗi câu thành:

- danh sách token ID

Quy tắc:
- từ không có trong embedding → bỏ qua
- nếu câu sau khi lọc rỗng → loại bỏ sample đó

⚠️ Điều này làm kết quả khác chương 7.

Ví dụ dữ liệu:
```
{
 'text': 'contains no wit , only labored gags',
 'label': tensor([0.]),
 'input_ids': tensor([3475, 87, 15888, 90, 27695, 42637])
}
```

In [56]:
import pandas as pd

train_df = pd.read_csv("SST-2/train.tsv",sep="\t")
dev_df = pd.read_csv("SST-2/dev.tsv",sep="\t")

In [57]:
def text_to_ids(text):
    ids = []

    for word in text.split():
        if word in word2id:
            ids.append(word2id[word])

    return ids

# test def
text_to_ids("contains no wit , only labored gags")

[7650, 6604, 11500, 8603, 7234, 12444]

In [58]:
# train

import torch

train_data = []

for _, row in train_df.iterrows():
    text = row["sentence"]

    label = float(row["label"])

    ids = text_to_ids(text)

    if len(ids) == 0:
        continue

    sample = {
        "text": text,
        "label": torch.tensor([label]),
        "input_ids": torch.tensor(ids)
    }

    train_data.append(sample)

In [59]:
# dev

dev_data = []

for _, row in dev_df.iterrows():
    text = row["sentence"]

    label = float(row["label"])

    ids = text_to_ids(text)

    if len(ids) == 0:
        continue

    sample = {
        "text": text,
        "label": torch.tensor([label]),
        "input_ids": torch.tensor(ids)
    }

    dev_data.append(sample)

In [60]:
print(train_data[0])
print(dev_data[0])

{'text': 'hide new secretions from the parental units ', 'label': tensor([0.]), 'input_ids': tensor([ 6897,  2621,  1614,  4245, 11985,  5490,  6954])}
{'text': "it 's a charming and often affecting journey . ", 'label': tensor([1.]), 'input_ids': tensor([10542,  4219, 10302, 12181, 10859])}


In [61]:
print(len(train_df))
print(len(train_data))

print(len(dev_df))
print(len(dev_data))

67349
66650
872
872


## 72. Bag of wordsモデルの構築

単語埋め込みの平均ベクトルでテキストの特徴ベクトルを表現し、重みベクトルとの内積でポジティブ及びネガティブを分類するニューラルネットワーク（ロジスティック回帰モデル）を設計せよ。

## 72. Xây dựng mô hình Bag of Words bằng NN

Hãy xây dựng mô hình:

- lấy embedding lookup
- tính trung bình vector
	​


Sau đó:

- đưa qua linear layer
- phân loại positive/negative

In [62]:
# try train_data[0]

test = train_data[0]
print(test)

input_ids = test["input_ids"]
print(input_ids)

embedding = embedding_matrix[input_ids]
print(embedding)

embedding_mean = embedding.mean(dim=0)
#print(embedding_mean)
print(embedding_mean.shape)


{'text': 'hide new secretions from the parental units ', 'label': tensor([0.]), 'input_ids': tensor([ 6897,  2621,  1614,  4245, 11985,  5490,  6954])}
tensor([ 6897,  2621,  1614,  4245, 11985,  5490,  6954])
tensor([[ 0.1885,  0.0552, -0.2148,  ..., -0.2285,  0.0767,  0.0239],
        [ 0.0113,  0.0289,  0.0835,  ...,  0.0815, -0.0459, -0.0464],
        [ 0.1895,  0.1045, -0.1973,  ..., -0.1768, -0.0608,  0.1396],
        ...,
        [ 0.0801,  0.1050,  0.0498,  ...,  0.0037,  0.0476, -0.0688],
        [ 0.2168, -0.1079, -0.1719,  ...,  0.2480,  0.0113,  0.1523],
        [ 0.1406, -0.0659,  0.2188,  ..., -0.1982,  0.0781,  0.0339]])
torch.Size([300])


In [63]:
# train

train_embeddings = []

for sample in train_data:
    input_ids = sample["input_ids"]

    embedding = embedding_matrix[input_ids]

    train_embeddings.append(embedding.mean(dim=0))

train_embeddings = torch.stack(train_embeddings)
train_embeddings.shape

torch.Size([66650, 300])

In [64]:
# dev

dev_embeddings = []

for sample in dev_data:
    input_ids = sample["input_ids"]

    embedding = embedding_matrix[input_ids]

    dev_embeddings.append(embedding.mean(dim=0))

dev_embeddings = torch.stack(dev_embeddings)
dev_embeddings.shape

torch.Size([872, 300])

In [65]:
# classify
train_labels = torch.stack(
    [sample["label"] for sample in train_data]
)

dev_labels = torch.stack(
    [sample["label"] for sample in dev_data]
)

X_train = train_embeddings
y_train = train_labels

X_dev = dev_embeddings
y_dev = dev_labels

print(X_train.shape)
print(y_train.shape)

torch.Size([66650, 300])
torch.Size([66650, 1])


In [66]:
import torch
import torch.nn as nn

class SentimentClassifier(nn.Module):

    def __init__(self, input_dim=300):
        super().__init__()

        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):

        logits = self.linear(x)

        return logits

In [67]:
model = SentimentClassifier()

logits = model(X_train[:5])
probs = torch.sigmoid(logits)

preds = (probs >= 0.5).int()

print("Probabilities:")
print(probs)

print("\nPredictions:")
print(preds)

Probabilities:
tensor([[0.5145],
        [0.5197],
        [0.5127],
        [0.5081],
        [0.5176]], grad_fn=<SigmoidBackward0>)

Predictions:
tensor([[1],
        [1],
        [1],
        [1],
        [1]], dtype=torch.int32)


## 73. モデルの学習

問題72で設計したモデルの重みベクトルを訓練セット上で学習せよ。ただし、学習中は単語埋め込み行列の値を固定せよ（単語埋め込み行列のファインチューニングは行わない）。また、学習時に損失値を表示するなど、学習の進捗状況をモニタリングできるようにせよ。

## 73. Huấn luyện mô hình

Hãy huấn luyện mô hình ở bài 72.

Quy tắc quan trọng:
- embedding phải cố định (freeze)

∇W=0

- chỉ update classifier weights
- theo dõi loss trong quá trình train

In [68]:
print(X_train.shape)
print(y_train.shape)

torch.Size([66650, 300])
torch.Size([66650, 1])


In [69]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [70]:
num_epochs = 20

for epoch in range(num_epochs):

    logits = model(X_train)

    loss = criterion(logits,y_train)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    print(
        f"Epoch {epoch+1}: "
        f"Loss = {loss.item():.4f}"
    )

Epoch 1: Loss = 0.6907
Epoch 2: Loss = 0.6895
Epoch 3: Loss = 0.6884
Epoch 4: Loss = 0.6872
Epoch 5: Loss = 0.6861
Epoch 6: Loss = 0.6850
Epoch 7: Loss = 0.6839
Epoch 8: Loss = 0.6828
Epoch 9: Loss = 0.6817
Epoch 10: Loss = 0.6806
Epoch 11: Loss = 0.6795
Epoch 12: Loss = 0.6785
Epoch 13: Loss = 0.6774
Epoch 14: Loss = 0.6764
Epoch 15: Loss = 0.6753
Epoch 16: Loss = 0.6743
Epoch 17: Loss = 0.6732
Epoch 18: Loss = 0.6722
Epoch 19: Loss = 0.6712
Epoch 20: Loss = 0.6702


## 74. モデルの評価

問題73で学習したモデルの開発セットにおける正解率を求めよ。

## 74. Đánh giá mô hình

Tính:

- accuracy trên dev set

In [71]:
model.eval()

with torch.no_grad():
    logits = model(X_dev)
    probs = torch.sigmoid(logits)

    preds = (probs >= 0.5).float()
    correct = (preds == y_dev).sum().item()
    total = y_dev.size(0)

    accuracy = correct / total

print(f"Correct: {correct}/{total}")
print(f"Accuracy: {accuracy:.4f}")

Correct: 444/872
Accuracy: 0.5092


## 75. パディング

複数の事例が与えられたとき、これらをまとめて一つのテンソル・オブジェクトで表現する関数`collate`を実装せよ。与えられた複数の事例のトークン列の長さが異なるときは、トークン列の長さが最も長いものに揃え、0番のトークンIDでパディングをせよ。さらに、トークン列の長さが長いものから順に、事例を並び替えよ。

例えば、訓練データセットの冒頭の4事例が次のように表されているとき、

```
[{'text': 'hide new secretions from the parental units',
  'label': tensor([0.]),
  'input_ids': tensor([  5785,     66, 113845,     18,     12,  15095,   1594])},
 {'text': 'contains no wit , only labored gags',
  'label': tensor([0.]),
  'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])},
 {'text': 'that loves its characters and communicates something rather beautiful about human nature',
  'label': tensor([1.]),
  'input_ids': tensor([    4,  5053,    45,  3305, 31647,   348,   904,  2815,    47,  1276,  1964])},
 {'text': 'remains utterly satisfied to remain the same throughout',
  'label': tensor([0.]),
  'input_ids': tensor([  987, 14528,  4941,   873,    12,   208,   898])}]
```

`collate`関数を通した結果は以下のようになることが想定される。

```
{'input_ids': tensor([
    [     4,   5053,     45,   3305,  31647,    348,    904,   2815,     47,   1276,   1964],
    [  5785,     66, 113845,     18,     12,  15095,   1594,      0,      0,      0,      0],
    [   987,  14528,   4941,    873,     12,    208,    898,      0,      0,      0,      0],
    [  3475,     87,  15888,     90,  27695,  42637,      0,      0,      0,      0,      0]]),
 'label': tensor([
    [1.],
    [0.],
    [0.],
    [0.]])}
```


## 75. Padding & collate function

Hãy viết hàm collate để gom nhiều sample thành batch tensor.

Quy tắc:
- pad bằng token ID = 0
- sequence dài nhất trong batch làm chuẩn

Ngoài ra:
- sắp xếp sample theo độ dài giảm dần

Ví dụ output:
```
tensor([
 [ 4, 5053, ..., 1964],
 [5785, 66, ...,   0],
 [987, 14528, ..., 0],
 [3475, 87, ..., 0]
])
```

In [72]:
# update model

class SentimentClassifier(nn.Module):

    def __init__(self, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True)

        self.linear = nn.Linear(embedding_matrix.shape[1], 1)

    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)

        mask = (input_ids != 0).unsqueeze(-1)
        embeddings = embeddings * mask

        lengths = mask.sum(dim=1)
        sentence_embeddings = embeddings.sum(dim=1) / lengths

        logits = self.linear(sentence_embeddings)


        return logits

In [73]:
def collate(batch):
    batch = sorted(batch, key=lambda x: len(x["input_ids"]), reverse=True)

    input_ids = [sample["input_ids"] for sample in batch]
    labels = [sample["label"] for sample in batch]

    max_len = max(len(ids) for ids in input_ids)
    padded_input_ids = torch.zeros((len(input_ids), max_len), dtype=torch.long)

    for i, ids in enumerate(input_ids):
        padded_input_ids[i, :len(ids)] = ids

    labels = torch.stack(labels)

    return {
        "input_ids": padded_input_ids,
        "label": labels
    }

samples = train_data[:5]
batch = collate(samples)

for row in batch["input_ids"]:
    print((row != 0).sum().item())

print(batch["input_ids"].shape)
print(batch["label"].shape)
print(batch)

11
9
7
7
6
torch.Size([5, 11])
torch.Size([5, 1])
{'input_ids': tensor([[  569,  1572, 12333,  5316,  1810, 11290,  7611,  6854,  4142,  3542,
             9],
        [ 6001, 11985,  1363, 12382, 11985,  7261,  9403, 11101, 11659,     0,
             0],
        [ 6897,  2621,  1614,  4245, 11985,  5490,  6954,     0,     0,     0,
             0],
        [ 2818, 10823,  6549, 12691, 11985,  9961, 11044,     0,     0,     0,
             0],
        [ 7650,  6604, 11500,  8603,  7234, 12444,     0,     0,     0,     0,
             0]]), 'label': tensor([[1.],
        [0.],
        [0.],
        [0.],
        [0.]])}


## 76. ミニバッチ学習

問題75のパディングの処理を活用して、ミニバッチでモデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

## 76. Mini-batch training

Sử dụng collate để:

- train bằng mini-batch
- đánh giá accuracy trên dev set

In [74]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_data,
    batch_size=8,
    shuffle=True,
    collate_fn=collate
)

dev_loader = DataLoader(
    dev_data,
    batch_size=8,
    shuffle=False,
    collate_fn=collate
)

In [75]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["label"].shape)

torch.Size([8, 21])
torch.Size([8, 1])


In [76]:
model = SentimentClassifier(embedding_matrix)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [78]:
epochs = 5

for epoch in range(epochs):
    model.train()

    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"]
        labels = batch["label"]

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"loss = {total_loss/len(train_loader):.4f}"
    )

Epoch 1: loss = 0.3769
Epoch 2: loss = 0.3725
Epoch 3: loss = 0.3705
Epoch 4: loss = 0.3692
Epoch 5: loss = 0.3684


In [79]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in dev_loader:

        input_ids = batch["input_ids"]
        labels = batch["label"]

        logits = model(input_ids)

        preds = (torch.sigmoid(logits) >= 0.5).float()

        correct += (preds == labels).sum().item()

        total += labels.size(0)

accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7993


## 77. GPU上での学習

問題76のモデル学習をGPU上で実行せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

## 77. Training trên GPU

Chạy mô hình trên GPU để:

- tăng tốc training
- tính accuracy dev set

In [80]:
import torch

print(torch.cuda.is_available())

print(torch.cuda.get_device_name(0))

True
Tesla T4


In [81]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [82]:
x = torch.randn(3,4)

print(x.device)

cpu


In [83]:
x = x.to(device)

print(x.device)

cuda:0


In [84]:
# đưa model lên gpu
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

model = model.to(device)
next(model.parameters()).device

cuda


device(type='cuda', index=0)

In [85]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [86]:
epochs = 5

for epoch in range(epochs):
    model.train()

    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"loss = {total_loss/len(train_loader):.4f}"
    )

Epoch 1: loss = 0.3680
Epoch 2: loss = 0.3675
Epoch 3: loss = 0.3673
Epoch 4: loss = 0.3670
Epoch 5: loss = 0.3668


In [87]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in dev_loader:

        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids)

        preds = (torch.sigmoid(logits) >= 0.5).float()

        correct += (preds == labels).sum().item()

        total += labels.size(0)

accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7982


## 78. 単語埋め込みのファインチューニング

問題77の学習において、単語埋め込みのパラメータも同時に更新するファインチューニングを導入せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

## 78. Fine-tuning embedding

Thay đổi setup:

- embedding được phép update

∇W != 0

Sau đó:

- train lại
- đánh giá dev accuracy

In [89]:
# update model - freeze = false

class SentimentClassifier(nn.Module):

    def __init__(self, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)

        self.linear = nn.Linear(embedding_matrix.shape[1], 1)

    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)

        mask = (input_ids != 0).unsqueeze(-1)
        embeddings = embeddings * mask

        lengths = mask.sum(dim=1)
        sentence_embeddings = embeddings.sum(dim=1) / lengths

        logits = self.linear(sentence_embeddings)


        return logits

In [90]:
model = SentimentClassifier(embedding_matrix)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

model = model.to(device)
next(model.parameters()).device

cuda


device(type='cuda', index=0)

In [91]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [92]:
epochs = 5

for epoch in range(epochs):
    model.train()

    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"loss = {total_loss/len(train_loader):.4f}"
    )

Epoch 1: loss = 0.3427
Epoch 2: loss = 0.2468
Epoch 3: loss = 0.2222
Epoch 4: loss = 0.2079
Epoch 5: loss = 0.1997


In [93]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in dev_loader:

        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids)

        preds = (torch.sigmoid(logits) >= 0.5).float()

        correct += (preds == labels).sum().item()

        total += labels.size(0)

accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7947


## 79. アーキテクチャの変更

ニューラルネットワークのアーキテクチャを自由に変更し、モデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。例えば、テキストの特徴ベクトル（単語埋め込みの平均ベクトル）に対して多層のニューラルネットワークを通したり、畳み込みニューラルネットワーク（CNN; Convolutional Neural Network）や再帰型ニューラルネットワーク（RNN; Recurrent Neural Network）などのモデルの学習に挑戦するとよい。

## 79. Thay đổi kiến trúc model

Hãy thử các kiến trúc khác nhau:

1. Deep neural network
nhiều hidden layers
2. CNN cho text
Convolutional Neural Network
3. RNN
Recurrent Neural Network

Mục tiêu:

- so sánh accuracy giữa các kiến trúc

### MLP

In [96]:
class SentimentClassifier(nn.Module):

    def __init__(self, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)

        self.fc1 = nn.Linear(embedding_matrix.shape[1], 128)

        self.reLU = nn.ReLU()

        self.fc2 = nn.Linear(128, 1)

    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)

        mask = (input_ids != 0).unsqueeze(-1)
        embeddings = embeddings * mask

        lengths = mask.sum(dim=1).float()
        sentence_embeddings = embeddings.sum(dim=1) / lengths

        x = self.fc1(sentence_embeddings)

        x = self.reLU(x)

        logits = self.fc2(x)

        return logits

In [97]:
model = SentimentClassifier(embedding_matrix)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

model = model.to(device)
next(model.parameters()).device

cuda


device(type='cuda', index=0)

In [98]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [99]:
epochs = 5

for epoch in range(epochs):
    model.train()

    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"loss = {total_loss/len(train_loader):.4f}"
    )

Epoch 1: loss = 0.3024
Epoch 2: loss = 0.2002
Epoch 3: loss = 0.1561
Epoch 4: loss = 0.1285
Epoch 5: loss = 0.1072


In [100]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in dev_loader:

        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids)

        preds = (torch.sigmoid(logits) >= 0.5).float()

        correct += (preds == labels).sum().item()

        total += labels.size(0)

accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7970


### CNN

In [102]:
class SentimentClassifier(nn.Module):

    def __init__(self, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False, padding_idx=0)

        self.conv = nn.Conv1d(in_channels=embedding_matrix.shape[1], out_channels=100, kernel_size=3)

        self.relu = nn.ReLU()

        self.linear = nn.Linear(100, 1)


    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        embeddings = embeddings.transpose(1, 2)

        x = self.conv(embeddings)

        x = self.relu(x)

        x = torch.max(x, dim=2).values

        logits = self.linear(x)

        return logits

In [103]:
model = SentimentClassifier(embedding_matrix)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

model = model.to(device)
next(model.parameters()).device

cuda


device(type='cuda', index=0)

In [104]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [105]:
epochs = 5

for epoch in range(epochs):
    model.train()

    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"loss = {total_loss/len(train_loader):.4f}"
    )

Epoch 1: loss = 0.2640
Epoch 2: loss = 0.1470
Epoch 3: loss = 0.1092
Epoch 4: loss = 0.0885
Epoch 5: loss = 0.0755


In [106]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in dev_loader:

        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids)

        preds = (torch.sigmoid(logits) >= 0.5).float()

        correct += (preds == labels).sum().item()

        total += labels.size(0)

accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7970


### RNN

In [107]:
class SentimentClassifier(nn.Module):

    def __init__(self, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False, padding_idx=0)

        self.rnn = nn.RNN(input_size=embedding_matrix.shape[1], hidden_size=128, batch_first=True)

        self.linear = nn.Linear(128, 1)


    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)

        output, hidden = self.rnn(embeddings)

        logits = self.linear(hidden[-1])

        return logits

In [108]:
model = SentimentClassifier(embedding_matrix)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

model = model.to(device)
next(model.parameters()).device

cuda


device(type='cuda', index=0)

In [109]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [110]:
epochs = 5

for epoch in range(epochs):
    model.train()

    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"loss = {total_loss/len(train_loader):.4f}"
    )

Epoch 1: loss = 0.6783
Epoch 2: loss = 0.6047
Epoch 3: loss = 0.4236
Epoch 4: loss = 0.3951
Epoch 5: loss = 0.3624


In [111]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in dev_loader:

        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids)

        preds = (torch.sigmoid(logits) >= 0.5).float()

        correct += (preds == labels).sum().item()

        total += labels.size(0)

accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7615


### LSTM

In [112]:
class SentimentClassifier(nn.Module):

    def __init__(self, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False, padding_idx=0)

        self.lstm = nn.LSTM(input_size=embedding_matrix.shape[1], hidden_size=128, batch_first=True)

        self.linear = nn.Linear(128, 1)


    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)

        output, (hidden, cell) = self.lstm(embeddings)

        logits = self.linear(hidden[-1])

        return logits

In [113]:
model = SentimentClassifier(embedding_matrix)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

model = model.to(device)
next(model.parameters()).device

cuda


device(type='cuda', index=0)

In [114]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [115]:
epochs = 5

for epoch in range(epochs):
    model.train()

    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"loss = {total_loss/len(train_loader):.4f}"
    )

Epoch 1: loss = 0.2987
Epoch 2: loss = 0.1633
Epoch 3: loss = 0.1200
Epoch 4: loss = 0.0953
Epoch 5: loss = 0.0792


In [116]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in dev_loader:

        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids)

        preds = (torch.sigmoid(logits) >= 0.5).float()

        correct += (preds == labels).sum().item()

        total += labels.size(0)

accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.8234
